# Notebook 05 — XCO₂ Retrieval

## Overview
This notebook demonstrates a complete **XCO₂ retrieval simulation** — mimicking the core algorithm used by satellite missions such as OCO-2, MicroCarb, and GOSAT.

## Retrieval Framework: Optimal Estimation

We minimise the cost function:

$$J(\xi) = \underbrace{\|\mathbf{y} - \mathbf{F}(\xi)\|^2_{\mathbf{S}_{\varepsilon}}}_{\text{spectral fit}} + \underbrace{(\xi - \xi_a)^2 / s_a^2}_{\text{prior constraint}}$$

where $\xi$ is a CO₂ **column scaling factor** (so XCO₂ = $\xi \cdot$ XCO₂$_\text{prior}$).

The **Levenberg-Marquardt** iterative scheme finds the optimal $\hat{\xi}$.

## XCO₂ Definition
$$\text{XCO}_2 = \frac{\int n_{CO_2}(z)\, dz}{\int n_{\text{dry}}(z)\, dz}  \quad [\text{ppm}]$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt

from hitran_model import hitran_cross_section, SYNTHETIC_CO2_LINES
from radiative_transfer import (
    forward_model, standard_atmosphere_profile,
    column_amount, air_mass_factor,
)
from retrieval import (
    compute_jacobian,
    optimal_estimation_scalar,
    iterative_retrieval,
    xco2_from_scaling,
    plot_retrieval_result,
    plot_xco2_sensitivity,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Setup — Forward Model and Observation

In [ ]:
# Spectral grid
nu = np.linspace(6210, 6270, 3000)

# Absorption cross section at surface conditions
xsec = hitran_cross_section(nu, SYNTHETIC_CO2_LINES, 101325.0, 288.0)

# Standard atmosphere at 420 ppm CO₂
xco2_prior_ppm = 420.0
profile  = standard_atmosphere_profile(co2_vmr_ppm=xco2_prior_ppm)
N_col_ref = column_amount(profile['number_density'], profile['layer_thickness_cm'])
print(f'Reference CO₂ column: {N_col_ref:.3e} molecules/cm²')

# Observation geometry
sza = 30.0   # solar zenith angle
vza = 0.0    # nadir satellite
amf = air_mass_factor(sza, vza)
print(f'Air-mass factor: {amf:.3f}')

# Forward model function (takes scaling factor ξ)
def fwd(xi):
    N = N_col_ref * xi
    return forward_model(nu, xsec, N, amf,
                         surface_albedo=0.25,
                         solar_zenith_deg=sza)['radiance']

## 2. Jacobian / Weighting Function

The Jacobian $K(\nu) = \partial I(\nu)/\partial \xi$ shows which spectral channels are most sensitive to CO₂.

In [ ]:
I_prior = fwd(1.0)
K = compute_jacobian(nu, I_prior, xsec, N_col_ref, amf)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

ax1.plot(nu, I_prior, color='darkorange', lw=1.5)
ax1.set_ylabel('Radiance (normalised)'); ax1.set_title('A priori Spectrum')
ax1.grid(True, alpha=0.3)

ax2.plot(nu, K, color='royalblue', lw=1.2)
ax2.fill_between(nu, K, 0, where=(K < 0), alpha=0.2, color='royalblue')
ax2.set_xlabel('Wavenumber [cm⁻¹]')
ax2.set_ylabel('∂I/∂ξ'); ax2.set_title('Jacobian K(ν) — Sensitivity to CO₂ Scaling')
ax2.grid(True, alpha=0.3)

plt.suptitle('Radiance and Jacobian', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/05a_jacobian.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Single-Step Optimal Estimation

For a small perturbation (within the linear regime), the linear OE solution works well.

In [ ]:
# Simulate observations at different 'true' XCO₂ values
xco2_true_values = [390, 410, 420, 430, 450, 480]
snr = 200.0
sigma_noise = I_prior.max() / snr

results_oe = []
for xco2_true in xco2_true_values:
    xi_true = xco2_true / xco2_prior_ppm
    y_obs   = fwd(xi_true) + np.random.normal(0, sigma_noise, len(nu))
    xi_hat, sigma_post, dfs = optimal_estimation_scalar(
        y_obs, I_prior, K, sigma_noise, xa=1.0, sa=0.15
    )
    xco2_ret, xco2_unc = xco2_from_scaling(xi_hat, xco2_prior_ppm, sigma_post)
    results_oe.append((xco2_true, xco2_ret, xco2_unc, dfs))

print(f'{"True [ppm]":>12}  {"Retrieved [ppm]":>15}  {"Uncertainty":>12}  {"DFS":>6}')
print('-' * 52)
for xco2_true, xco2_ret, xco2_unc, dfs in results_oe:
    print(f'{xco2_true:>12.1f}  {xco2_ret:>15.2f}  {xco2_unc:>12.2f}  {dfs:>6.3f}')

## 4. Iterative Levenberg-Marquardt Retrieval

For a larger perturbation from the prior, the iterative solver is more accurate.

In [ ]:
# True XCO₂ = 437 ppm  (significant departure from prior)
xco2_true_ppm = 437.0
xi_true       = xco2_true_ppm / xco2_prior_ppm

np.random.seed(42)
y_obs = fwd(xi_true) + np.random.normal(0, sigma_noise, len(nu))

result = iterative_retrieval(
    nu, y_obs, fwd,
    xa=1.0, sa=0.10,
    sigma_noise=sigma_noise,
    max_iter=20, gamma=10.0,
)

xi_hat     = result['xi_hat']
sigma_post = result['sigma_post']
xco2_ret, xco2_unc = xco2_from_scaling(xi_hat, xco2_prior_ppm, sigma_post)

print('\n=== Iterative Retrieval Result ===')
print(f'  True XCO₂   : {xco2_true_ppm:.1f} ppm')
print(f'  Prior XCO₂  : {xco2_prior_ppm:.1f} ppm')
print(f'  Retrieved   : {xco2_ret:.2f} ± {xco2_unc:.2f} ppm')
print(f'  Error       : {xco2_ret - xco2_true_ppm:+.2f} ppm')
print(f'  Iterations  : {result["n_iter"]}')
print(f'  ξ̂  = {xi_hat:.5f}   (true ξ = {xi_true:.5f})')

I_retrieved = fwd(xi_hat)
plot_retrieval_result(
    nu, y_obs, I_prior, I_retrieved,
    xco2_true=xco2_true_ppm,
    xco2_prior=xco2_prior_ppm,
    xco2_ret=xco2_ret,
    xco2_unc=xco2_unc,
    cost_history=result['cost_history'],
    savefig='../figures/05b_retrieval_result.png',
)

## 5. Retrieval Error Statistics

Run 200 simulations with random noise to characterise the retrieval performance.

In [ ]:
np.random.seed(0)
N_trials = 200
errors   = []

for _ in range(N_trials):
    y_noisy = fwd(xi_true) + np.random.normal(0, sigma_noise, len(nu))
    xi_hat_i, sp_i, _ = optimal_estimation_scalar(
        y_noisy, I_prior, K, sigma_noise, xa=1.0, sa=0.10)
    xco2_i, _ = xco2_from_scaling(xi_hat_i, xco2_prior_ppm, sp_i)
    errors.append(xco2_i - xco2_true_ppm)

errors = np.array(errors)
print(f'Bias  : {errors.mean():.3f} ppm')
print(f'1-σ   : {errors.std():.3f} ppm')
print(f'RMS   : {np.sqrt((errors**2).mean()):.3f} ppm')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(errors, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=1.5, ls='--', label='Zero error')
ax.axvline(errors.mean(), color='tomato', lw=1.5, label=f'Bias = {errors.mean():.2f} ppm')
ax.set_xlabel('XCO₂ retrieval error [ppm]')
ax.set_ylabel('Count')
ax.set_title(f'Retrieval Error Statistics (N={N_trials}, SNR={snr:.0f})\n'
             f'σ = {errors.std():.2f} ppm')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/05c_error_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Spectral Radiance Sensitivity to XCO₂

In [ ]:
plot_xco2_sensitivity(
    nu, fwd,
    xco2_values_ppm=[370, 400, 420, 450, 480],
    xco2_ref_ppm=420.0,
    savefig='../figures/05d_radiance_sensitivity.png',
)

## Summary

This notebook demonstrated the full end-to-end CO₂ retrieval chain:

1. **Forward model** — maps XCO₂ (via a column scaling factor ξ) to a simulated radiance spectrum.
2. **Jacobian** — identifies spectral channels most sensitive to CO₂.
3. **Optimal estimation** — balances spectral fit against prior information to retrieve XCO₂.
4. **Iterative solver** — handles non-linearities for larger CO₂ departures.
5. **Error statistics** — characterises retrieval noise and bias.

Real satellite retrievals (OCO-2, MicroCarb) extend this framework with:
- Simultaneous retrieval of surface pressure, albedo, aerosol, and water vapour
- Multi-band (O₂-A, weak CO₂, strong CO₂) observations
- Full covariance matrices and profile-level state vectors
- Careful treatment of scattering by aerosols and clouds